# Phase 3


In [ ]:
# Clean old installs first
#!pip uninstall -y codecarbon transformers accelerate sentencepiece huggingface_hub tree_sitter tree_sitter_languages pandas tqdm rapidfuzz

# Stable install set
!pip install --no-cache-dir codecarbon==2.3.3
!pip install --no-cache-dir "transformers<5" accelerate sentencepiece huggingface_hub
!pip install --no-cache-dir tree_sitter tree_sitter_languages
!pip install --no-cache-dir pandas tqdm rapidfuzz

In [ ]:
from codecarbon import EmissionsTracker
print("CodeCarbon import success")

In [ ]:
import os, re, json, time, hashlib
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tree_sitter_languages import get_parser, get_language
from codecarbon import EmissionsTracker
from huggingface_hub import login

# Use your own token securely
login("hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
BASE_DIR = "/content/drive/MyDrive/All_Code_Files"
OUT_DIR  = "/content/drive/MyDrive/PHASE3_File4_TestCases"

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

NUM_RUNS = 1
CPU_WARMUP_SECS = 300
GPU_WARMUP_SECS = 300
COOLDOWN_SECS = 60
MEASURE_POWER_SECS = 1

INCLUDE_GENERATED = False
MAX_CONTEXT_CHARS = 18000

MIN_TESTS_PER_METHOD = 1
MAX_TESTS_PER_FILE = 60

# file extensions considered "code"
CODE_EXTS = {
    ".java", ".kt", ".kts",
    ".py",
    ".js", ".ts", ".jsx", ".tsx",
    ".go", ".rs",
    ".cpp", ".cc", ".c", ".h", ".hpp",
    ".cs",
    ".php", ".rb",
    ".swift",
    ".xml",   # added
}

IGNORE_DIR_NAMES = {
    ".git", ".svn", ".hg",
    "node_modules", "dist", "out",
    ".idea", ".gradle",
    "__pycache__", ".pytest_cache", ".mypy_cache",
    "venv", ".venv", "env", ".env",
    "target", "bin", "obj",
}

GENERATED_DIR_HINTS = {
    "build", "generated", "intermediates", ".cxx", "cmake-build", "coverage", "tmp"
}

MODEL_TAG = MODEL_ID.replace("/", "__")
PHASE3_ROOT = os.path.join(OUT_DIR, MODEL_TAG)
os.makedirs(PHASE3_ROOT, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("PHASE3_ROOT:", PHASE3_ROOT)

In [ ]:
def backup_if_exists(path):
    if os.path.exists(path):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        new_name = f"{path}.old_{ts}"
        os.rename(path, new_name)
        print(f"Existing file backed up: {new_name}")

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def make_tracker(output_dir: str, project_name: str, output_file: str):
    os.makedirs(output_dir, exist_ok=True)
    backup_if_exists(os.path.join(output_dir, output_file))
    return EmissionsTracker(
        project_name=project_name,
        output_dir=output_dir,
        output_file=output_file,
        measure_power_secs=MEASURE_POWER_SECS,
        log_level="warning",
    )

def is_ignored_dir(path: str) -> bool:
    parts = set(os.path.normpath(path).split(os.sep))
    if parts & IGNORE_DIR_NAMES:
        return True
    if not INCLUDE_GENERATED and (parts & GENERATED_DIR_HINTS):
        return True
    return False

def safe_read_text(fp: str, max_bytes: Optional[int] = None) -> str:
    with open(fp, "rb") as f:
        data = f.read() if max_bytes is None else f.read(max_bytes)
    for enc in ("utf-8", "utf-16", "latin-1"):
        try:
            return data.decode(enc)
        except:
            pass
    return data.decode("utf-8", errors="ignore")

def relpath(fp: str, root: str) -> str:
    try:
        return os.path.relpath(fp, root)
    except:
        return fp

def looks_like_resource_folder(p: str) -> bool:
    norm = p.replace("\\", "/").lower()
    return "/res/" in norm or norm.endswith("/res") or "/drawable" in norm or "/layout" in norm

def tree_summary(root: str, max_lines: int = 220) -> str:
    lines = []
    allowed = {
        ".java", ".kt", ".py", ".js", ".ts", ".jsx", ".tsx",
        ".xml", ".json", ".yml", ".yaml", ".gradle", ".kts", ".md"
    }
    for dirpath, dirnames, filenames in os.walk(root):
        if is_ignored_dir(dirpath):
            dirnames[:] = []
            continue
        depth = len(os.path.relpath(dirpath, root).split(os.sep))
        indent = "  " * max(depth, 0)
        show_files = [f for f in filenames if os.path.splitext(f)[1].lower() in allowed]
        show_files = sorted(show_files)[:10]
        lines.append(f"{indent}{os.path.basename(dirpath)}/")
        for f in show_files:
            lines.append(f"{indent}  - {f}")
        if len(lines) >= max_lines:
            break
    return "\n".join(lines[:max_lines])

def collect_code_files(root: str) -> List[str]:
    files = []
    for dirpath, dirnames, filenames in os.walk(root):
        if is_ignored_dir(dirpath):
            dirnames[:] = []
            continue
        for fn in filenames:
            ext = os.path.splitext(fn)[1].lower()
            if ext in CODE_EXTS:
                files.append(os.path.join(dirpath, fn))
    files.sort()
    return files

def guess_language_from_ext(ext: str) -> Optional[str]:
    ext = ext.lower()
    if ext == ".java": return "java"
    if ext in (".kt", ".kts"): return "kotlin"
    if ext == ".py": return "python"
    if ext in (".js", ".jsx"): return "javascript"
    if ext in (".ts", ".tsx"): return "typescript"
    if ext == ".xml": return "xml"
    return None

In [ ]:
def find_code_units(base_dir: str) -> List[Tuple[str, str]]:
    units = []
    seen = set()

    for dirpath, dirnames, filenames in os.walk(base_dir):
        if is_ignored_dir(dirpath):
            dirnames[:] = []
            continue

        if looks_like_resource_folder(dirpath):
            continue

        low = dirpath.replace("\\", "/").lower()

        is_source_root = (
            "/src/main/java" in low or "/src/main/kotlin" in low or
            "/src/" in low or
            low.endswith("/src") or low.endswith("/app") or low.endswith("/lib") or
            low.endswith("/server") or low.endswith("/backend") or low.endswith("/frontend")
        )

        if not is_source_root:
            continue

        has_code = any(os.path.splitext(f)[1].lower() in CODE_EXTS for f in filenames)
        if not has_code:
            try:
                sub_has_code = False
                for sd in dirnames[:30]:
                    sp = os.path.join(dirpath, sd)
                    try:
                        for f in os.listdir(sp):
                            if os.path.isfile(os.path.join(sp, f)) and os.path.splitext(f)[1].lower() in CODE_EXTS:
                                sub_has_code = True
                                break
                    except:
                        pass
                    if sub_has_code:
                        break
                has_code = sub_has_code
            except:
                pass

        if has_code:
            unit_name = os.path.basename(os.path.normpath(dirpath)) or "unit"
            key = (unit_name, dirpath)
            if key not in seen:
                units.append((unit_name, dirpath))
                seen.add(key)

    if not units:
        any_code = False
        for dirpath, dirnames, filenames in os.walk(base_dir):
            if is_ignored_dir(dirpath):
                dirnames[:] = []
                continue
            if any(os.path.splitext(f)[1].lower() in CODE_EXTS for f in filenames):
                any_code = True
                break
        if any_code:
            units = [(os.path.basename(os.path.normpath(base_dir)) or "root", base_dir)]

    units = [(n, p) for (n, p) in units if not looks_like_resource_folder(p)]
    units.sort(key=lambda x: (x[0].lower(), x[1].lower()))
    return units

In [ ]:
@dataclass
class Symbol:
    language: str
    kind: str
    name: str
    signature: str
    file: str
    start_line: int
    end_line: int
    modifiers: List[str]
    is_testable: bool
    evidence: str

PARSER_CACHE = {}
LANG_CACHE = {}

def get_ts_parser_cached(lang: str):
    if lang not in PARSER_CACHE:
        try:
            PARSER_CACHE[lang] = get_parser(lang)
        except Exception:
            PARSER_CACHE[lang] = None
    return PARSER_CACHE[lang]

def get_ts_language_cached(lang: str):
    if lang not in LANG_CACHE:
        try:
            LANG_CACHE[lang] = get_language(lang)
        except Exception:
            LANG_CACHE[lang] = None
    return LANG_CACHE[lang]

def node_text(src: bytes, node) -> str:
    return src[node.start_byte:node.end_byte].decode("utf-8", errors="ignore")

def line_of_byte(src: bytes, b: int) -> int:
    return src[:b].count(b"\n") + 1

TS_QUERIES = {
    "java": r"""
      (class_declaration name: (identifier) @class.name) @class.node
      (interface_declaration name: (identifier) @class.name) @class.node
      (enum_declaration name: (identifier) @class.name) @class.node
      (method_declaration name: (identifier) @method.name) @method.node
      (constructor_declaration name: (identifier) @method.name) @method.node
    """,
    "kotlin": r"""
      (class_declaration name: (simple_identifier) @class.name) @class.node
      (object_declaration name: (simple_identifier) @class.name) @class.node
      (function_declaration name: (simple_identifier) @method.name) @method.node
    """,
    "python": r"""
      (class_definition name: (identifier) @class.name) @class.node
      (function_definition name: (identifier) @method.name) @method.node
    """,
    "javascript": r"""
      (class_declaration name: (identifier) @class.name) @class.node
      (function_declaration name: (identifier) @method.name) @method.node
      (method_definition name: (property_identifier) @method.name) @method.node
    """,
    "typescript": r"""
      (class_declaration name: (type_identifier) @class.name) @class.node
      (function_declaration name: (identifier) @method.name) @method.node
      (method_definition name: (property_identifier) @method.name) @method.node
    """,
}

def _captures_compat(language, root_node, query_str: str):
    query = language.query(query_str)
    try:
        return query.captures(root_node)
    except Exception:
        pass
    try:
        from tree_sitter import QueryCursor
        cursor = QueryCursor()
        return list(cursor.captures(query, root_node))
    except Exception:
        return []

def is_testable_default(name: str) -> bool:
    if not name or name == "unknown":
        return False
    return True

In [ ]:
def regex_extract_java_symbols(code: str, rel_file: str) -> List[Symbol]:
    syms = []
    for m in re.finditer(r"\b(class|interface|enum)\s+([A-Za-z_]\w*)", code):
        name = m.group(2)
        start = code[:m.start()].count("\n") + 1
        syms.append(Symbol("java", "class", name, name, rel_file, start, start, [], False, f"{m.group(1)} {name}"))

    method_re = re.compile(
        r"""
        (?P<mods>(?:public|private|protected|static|final|abstract|synchronized|native|strictfp|\s)+)?
        (?:(?P<ret>[A-Za-z_]\w*(?:<[^;>{}()]*>)?(?:\[\])*)\s+)?
        (?P<name>[A-Za-z_]\w*)\s*
        \((?P<params>[^)]*)\)\s*
        (?:throws\s+[A-Za-z0-9_,\s]+)?\s*
        \{
        """,
        re.VERBOSE
    )
    banned = {"if","for","while","switch","catch","return","new","do","try","else","case","throw","synchronized"}
    for m in method_re.finditer(code):
        name = m.group("name")
        if name in banned:
            continue
        sig = code[m.start(): m.start()+260].replace("\n"," ").strip()
        mods_txt = (m.group("mods") or "")
        mods = re.findall(r"\b(public|private|protected|static|final|abstract)\b", mods_txt)
        start = code[:m.start()].count("\n") + 1
        syms.append(Symbol("java", "method", name, sig, rel_file, start, start, mods, is_testable_default(name), sig[:160]))
    return syms

def regex_extract_kotlin_symbols(code: str, rel_file: str) -> List[Symbol]:
    syms = []
    for m in re.finditer(r"\b(class|object)\s+([A-Za-z_]\w*)", code):
        name = m.group(2)
        start = code[:m.start()].count("\n") + 1
        syms.append(Symbol("kotlin", "class", name, name, rel_file, start, start, [], False, f"{m.group(1)} {name}"))
    for m in re.finditer(r"\bfun\s+([A-Za-z_]\w*)\s*\(", code):
        name = m.group(1)
        start = code[:m.start()].count("\n") + 1
        sig = code[m.start(): m.start()+260].replace("\n"," ").strip()
        syms.append(Symbol("kotlin", "method", name, sig, rel_file, start, start, [], is_testable_default(name), sig[:160]))
    return syms

def regex_extract_js_ts_symbols(code: str, rel_file: str, lang: str) -> List[Symbol]:
    syms = []

    for m in re.finditer(r"\bclass\s+([A-Za-z_]\w*)", code):
        name = m.group(1)
        start = code[:m.start()].count("\n") + 1
        syms.append(Symbol(lang, "class", name, name, rel_file, start, start, [], False, f"class {name}"))

    for m in re.finditer(
        r"\b(?:export\s+default\s+|export\s+|default\s+)?function\s+([A-Za-z_]\w*)\s*\(",
        code
    ):
        name = m.group(1)
        start = code[:m.start()].count("\n") + 1
        sig = code[m.start(): m.start()+260].replace("\n", " ").strip()
        syms.append(Symbol(lang, "function", name, sig, rel_file, start, start, [], is_testable_default(name), sig[:160]))

    for m in re.finditer(
        r"\b(?:export\s+)?(?:const|let|var)\s+([A-Za-z_]\w*)\s*=\s*(?:async\s*)?\([^)]*\)\s*(?::\s*[^=]+)?=>",
        code
    ):
        name = m.group(1)
        start = code[:m.start()].count("\n") + 1
        sig = code[m.start(): m.start()+260].replace("\n", " ").strip()
        syms.append(Symbol(lang, "function", name, sig, rel_file, start, start, [], is_testable_default(name), sig[:160]))

    for m in re.finditer(
        r"\b(?:export\s+)?(?:const|let|var)\s+([A-Za-z_]\w*)\s*=\s*(?:async\s*)?[A-Za-z_]\w*\s*(?::\s*[^=]+)?=>",
        code
    ):
        name = m.group(1)
        start = code[:m.start()].count("\n") + 1
        sig = code[m.start(): m.start()+260].replace("\n", " ").strip()
        syms.append(Symbol(lang, "function", name, sig, rel_file, start, start, [], is_testable_default(name), sig[:160]))

    for m in re.finditer(
        r"\b([A-Za-z_]\w*)\s*:\s*(?:async\s*)?\([^)]*\)\s*(?::\s*[^=]+)?=>",
        code
    ):
        name = m.group(1)
        start = code[:m.start()].count("\n") + 1
        sig = code[m.start(): m.start()+260].replace("\n", " ").strip()
        syms.append(Symbol(lang, "function", name, sig, rel_file, start, start, [], is_testable_default(name), sig[:160]))

    for m in re.finditer(
        r"\b([A-Za-z_]\w*)\s*:\s*(?:async\s*)?[A-Za-z_]\w*\s*(?::\s*[^=]+)?=>",
        code
    ):
        name = m.group(1)
        start = code[:m.start()].count("\n") + 1
        sig = code[m.start(): m.start()+260].replace("\n", " ").strip()
        syms.append(Symbol(lang, "function", name, sig, rel_file, start, start, [], is_testable_default(name), sig[:160]))

    for m in re.finditer(
        r"(?m)^\s*(?:public\s+|private\s+|protected\s+|static\s+|async\s+)*([A-Za-z_]\w*)\s*\([^;\n{}]*\)\s*\{",
        code
    ):
        name = m.group(1)
        banned = {"if", "for", "while", "switch", "catch", "constructor"}
        if name in banned:
            continue
        start = code[:m.start()].count("\n") + 1
        sig = code[m.start(): m.start()+260].replace("\n", " ").strip()
        syms.append(Symbol(lang, "method", name, sig, rel_file, start, start, [], is_testable_default(name), sig[:160]))

    uniq = {}
    for s in syms:
        key = (s.file, s.kind, s.name, s.start_line, s.signature[:120])
        uniq[key] = s
    return list(uniq.values())

In [ ]:
def extract_ui_actions_java_kotlin(code: str, rel_file: str, lang: str) -> List[Symbol]:
    out = []
    lines = code.splitlines()

    for i, ln in enumerate(lines, start=1):
        m = re.search(r"findViewById\s*\(\s*R\.id\.([A-Za-z_]\w*)\s*\)", ln)
        if m:
            comp_id = m.group(1)
            out.append(Symbol(lang, "ui_component", comp_id, "findViewById", rel_file, i, i, [], True, ln.strip()))

        if "setOnClickListener" in ln:
            out.append(Symbol(lang, "ui_action", "onClick", "setOnClickListener", rel_file, i, i, [], True, ln.strip()))
        if "setOnCheckedChangeListener" in ln or "setOnCheckedChanged" in ln:
            out.append(Symbol(lang, "ui_action", "onCheckedChange", "setOnCheckedChangeListener", rel_file, i, i, [], True, ln.strip()))
        if "addTextChangedListener" in ln:
            out.append(Symbol(lang, "ui_action", "onTextChanged", "addTextChangedListener", rel_file, i, i, [], True, ln.strip()))
        if ".setOnItemClickListener" in ln:
            out.append(Symbol(lang, "ui_action", "onItemClick", "setOnItemClickListener", rel_file, i, i, [], True, ln.strip()))
        if ".setOnLongClickListener" in ln:
            out.append(Symbol(lang, "ui_action", "onLongClick", "setOnLongClickListener", rel_file, i, i, [], True, ln.strip()))
    return out

def extract_ui_actions_js_ts(code: str, rel_file: str, lang: str) -> List[Symbol]:
    out = []
    lines = code.splitlines()

    event_patterns = [
        ("ui_action", "onPress", r"\bonPress\s*="),
        ("ui_action", "onClick", r"\bonClick\s*="),
        ("ui_action", "onChangeText", r"\bonChangeText\s*="),
        ("ui_action", "onChange", r"\bonChange\s*="),
        ("ui_action", "onSubmitEditing", r"\bonSubmitEditing\s*="),
        ("ui_action", "onLongPress", r"\bonLongPress\s*="),
    ]

    component_patterns = [
        ("ui_component", "Button", r"<Button\b"),
        ("ui_component", "TextInput", r"<TextInput\b"),
        ("ui_component", "TouchableOpacity", r"<TouchableOpacity\b"),
        ("ui_component", "Pressable", r"<Pressable\b"),
        ("ui_component", "Input", r"<Input\b"),
        ("ui_component", "Alert", r"<Alert\b"),
        ("ui_component", "View", r"<View\b"),
    ]

    for i, ln in enumerate(lines, start=1):
        for kind, name, pat in event_patterns:
            if re.search(pat, ln):
                out.append(Symbol(lang, kind, name, name, rel_file, i, i, [], True, ln.strip()))
        for kind, name, pat in component_patterns:
            if re.search(pat, ln):
                out.append(Symbol(lang, kind, name, name, rel_file, i, i, [], True, ln.strip()))

    for i, ln in enumerate(lines, start=1):
        if "useMutation(" in ln:
            out.append(Symbol(lang, "ui_action", "useMutation", "useMutation", rel_file, i, i, [], True, ln.strip()))
        if "useForm(" in ln:
            out.append(Symbol(lang, "ui_action", "useForm", "useForm", rel_file, i, i, [], True, ln.strip()))

    return out

def extract_xml_ui_components(xml_text: str, rel_file: str) -> List[Symbol]:
    out = []
    pattern = re.compile(
        r"<\s*([A-Za-z0-9_.]+)[^>]*android:id\s*=\s*\"@[\+\w]+/([A-Za-z0-9_]+)\"[^>]*>",
        re.IGNORECASE
    )
    for m in pattern.finditer(xml_text):
        widget = m.group(1)
        wid = m.group(2)
        start_byte = m.start()
        line = xml_text[:start_byte].count("\n") + 1
        out.append(Symbol("xml", "ui_component", wid, widget, rel_file, line, line, [], True, f"{widget} id={wid}"))
    return out

In [ ]:
def extract_symbols_from_file(fp: str, repo_root: str) -> List[Symbol]:
    ext = os.path.splitext(fp)[1].lower()
    lang = guess_language_from_ext(ext)
    text = safe_read_text(fp)
    rel = relpath(fp, repo_root)

    syms: List[Symbol] = []

    if ext == ".xml":
        syms.extend(extract_xml_ui_components(text, rel))
        return syms

    if not lang:
        return []

    parser = get_ts_parser_cached(lang)
    language = get_ts_language_cached(lang)

    if parser is not None and language is not None and lang in TS_QUERIES:
        try:
            src = text.encode("utf-8", errors="ignore")
            tree = parser.parse(src)
            root_node = tree.root_node
            caps = _captures_compat(language, root_node, TS_QUERIES[lang])

            for node, cap in caps:
                if cap == "class.name":
                    name = node_text(src, node)
                    parent = node.parent
                    syms.append(Symbol(
                        language=lang,
                        kind="class",
                        name=name,
                        signature=name,
                        file=rel,
                        start_line=line_of_byte(src, parent.start_byte),
                        end_line=line_of_byte(src, parent.end_byte),
                        modifiers=[],
                        is_testable=False,
                        evidence=f"class {name}"
                    ))
                elif cap == "method.name":
                    name = node_text(src, node)
                    parent = node.parent
                    sig = node_text(src, parent)[:260].replace("\n", " ").strip()
                    syms.append(Symbol(
                        language=lang,
                        kind="method" if lang in ("java", "kotlin", "javascript", "typescript") else "function",
                        name=name,
                        signature=sig,
                        file=rel,
                        start_line=line_of_byte(src, parent.start_byte),
                        end_line=line_of_byte(src, parent.end_byte),
                        modifiers=[],
                        is_testable=is_testable_default(name),
                        evidence=sig[:160]
                    ))
        except Exception:
            syms = []

    if not syms and lang == "java":
        syms = regex_extract_java_symbols(text, rel)
    elif not syms and lang == "kotlin":
        syms = regex_extract_kotlin_symbols(text, rel)

    if lang in ("javascript", "typescript"):
        syms.extend(regex_extract_js_ts_symbols(text, rel, lang))

    if lang in ("java", "kotlin"):
        syms.extend(extract_ui_actions_java_kotlin(text, rel, lang))

    if lang in ("javascript", "typescript"):
        syms.extend(extract_ui_actions_js_ts(text, rel, lang))

    uniq = {}
    for s in syms:
        key = (s.file, s.kind, s.name, s.start_line, s.end_line, s.signature[:80])
        uniq[key] = s
    return list(uniq.values())

In [ ]:
def file_api_map(fp: str, repo_root: str) -> Dict[str, Any]:
    rel = relpath(fp, repo_root)
    text = safe_read_text(fp)

    syms = extract_symbols_from_file(fp, repo_root)

    classes = [s for s in syms if s.kind == "class"]
    methods = [s for s in syms if s.kind in ("method", "function") and s.is_testable]
    ui_actions = [s for s in syms if s.kind in ("ui_action", "ui_component")]

    return {
        "file_path": fp,
        "rel_file": rel,
        "language": guess_language_from_ext(os.path.splitext(fp)[1]),
        "stats": {
            "classes": len(classes),
            "methods": len(methods),
            "ui": len(ui_actions),
        },
        "classes": [asdict(s) for s in classes],
        "methods": [asdict(s) for s in methods],
        "ui_actions": [asdict(s) for s in ui_actions],
        "code_preview": "\n".join(text.splitlines()[:220])
    }

def pack_context_for_file(api_map: Dict[str, Any], repo_root: str) -> str:
    rel_file = api_map["rel_file"]
    header = []
    header.append(f"FILE: {rel_file}")
    header.append(f"LANG: {api_map.get('language')}")
    header.append("\nCLASSES:")
    for c in api_map.get("classes", [])[:50]:
        header.append(f"- {c['name']} ({c['file']}:{c['start_line']}-{c['end_line']})")

    header.append("\nMETHODS (must be covered):")
    for m in api_map.get("methods", [])[:200]:
        header.append(f"- {m['name']} | {m['file']}:{m['start_line']}-{m['end_line']}")
        header.append(f"  sig: {m['signature'][:220]}")

    header.append("\nUI / CTA / INPUT SIGNALS:")
    for a in api_map.get("ui_actions", [])[:80]:
        header.append(f"- {a['kind']} {a['name']} | {a['file']}:{a['start_line']} | {a['evidence'][:220]}")

    header.append("\nCODE (snippet):")
    code = safe_read_text(api_map["file_path"])
    excerpt = code
    ctx = "\n".join(header)
    budget = MAX_CONTEXT_CHARS - len(ctx)
    if budget < 2000:
        budget = 2000
    if len(excerpt) > budget:
        excerpt = excerpt[:budget] + "\n...[trimmed]...\n"

    return ctx + "\n" + excerpt

In [ ]:
def prepare_tokenizer_and_model(tokenizer, model):
    if tokenizer.pad_token is None:
        if tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
        elif tokenizer.eos_token_id is not None:
            tokenizer.pad_token_id = tokenizer.eos_token_id

    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    tokenizer.padding_side = "left"

    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tokenizer.pad_token_id

    if hasattr(model, "generation_config"):
        if getattr(model.generation_config, "pad_token_id", None) is None:
            model.generation_config.pad_token_id = tokenizer.pad_token_id

    return tokenizer, model

def load_qwen(model_id: str):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype="auto",
        trust_remote_code=True
    )
    tok, model = prepare_tokenizer_and_model(tok, model)
    model.eval()
    return tok, model

In [ ]:
setup_tracking_dir = os.path.join(PHASE3_ROOT, "setup_tracking")

setup_tracker = make_tracker(
    output_dir=setup_tracking_dir,
    project_name="phase3_code_validation_setup",
    output_file="phase3_setup_emissions.csv"
)

print("PHASE 3 SETUP TRACKER START")
setup_tracker.start()
setup_t0 = time.time()

print(f"Loading model: {MODEL_ID}")
tokenizer, model = load_qwen(MODEL_ID)
print(f"Loaded model: {MODEL_ID}")

setup_emissions = setup_tracker.stop()
setup_duration = time.time() - setup_t0
print("PHASE 3 SETUP TRACKER STOP")

setup_summary = {
    "phase": "phase3_code_validation",
    "model_id": MODEL_ID,
    "setup_duration_sec": setup_duration,
    "setup_emissions_kg": setup_emissions,
    "device": device,
    "input_base_dir": BASE_DIR
}
save_json(setup_summary, os.path.join(setup_tracking_dir, "phase3_setup_summary.json"))

setup_summary

In [ ]:
def warmup_cpu(duration_secs):
    print(f"\nCPU warm-up for {duration_secs} seconds...")
    start = time.time()
    a, b = 0, 1
    while (time.time() - start) < duration_secs:
        a, b = b, a + b
        if a > 10**7:
            a, b = 0, 1
    print("CPU warm-up finished.\n")

@torch.no_grad()
def warmup_phase3_once():
    messages = [
        {"role": "system", "content": "You are a senior SDET."},
        {"role": "user", "content": "Generate one small test case in JSON array format for a login() method."}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    _ = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

def run_phase3_warmup():
    warmup_cpu(CPU_WARMUP_SECS)

    print(f"GPU/LLM warm-up for {GPU_WARMUP_SECS} seconds...")
    start = time.time()
    while (time.time() - start) < GPU_WARMUP_SECS:
        warmup_phase3_once()
        time.sleep(2)
    print("GPU/LLM warm-up finished.\n")

print("\n========== PHASE 3 WARM-UP START (EXCLUDED) ==========")
run_phase3_warmup()
print("========== PHASE 3 WARM-UP STOP ==========\n")

In [ ]:
def normalize_name(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip().lower()

def expected_method_names(api_map: Dict[str, Any]) -> List[str]:
    names = []
    for m in api_map.get("methods", []):
        n = (m.get("name") or "").strip()
        if n:
            names.append(n)
    return sorted(set(names), key=lambda x: x.lower())

def covered_method_names_from_items(items: List[Dict[str, Any]]) -> set:
    covered = set()
    for tc in (items or []):
        arr = tc.get("covered_methods", [])
        if isinstance(arr, list):
            for x in arr:
                if isinstance(x, str) and x.strip():
                    covered.add(x.strip())
    expanded = set()
    for x in covered:
        expanded.add(normalize_name(x))
        if "." in x:
            expanded.add(normalize_name(x.split(".")[-1]))
    return expanded

def find_missing_methods(api_map: Dict[str, Any], items: List[Dict[str, Any]]) -> List[str]:
    expected = expected_method_names(api_map)
    got = covered_method_names_from_items(items or [])
    missing = []
    for m in expected:
        if normalize_name(m) not in got:
            missing.append(m)
    return missing

def try_parse_json_array_strict(text: str) -> Optional[List[Dict[str, Any]]]:
    arrays = re.findall(r"\[\s*\{.*?\}\s*\]", text, flags=re.DOTALL)
    if not arrays:
        return None
    cand = arrays[-1]
    try:
        return json.loads(cand)
    except:
        cand2 = re.sub(r",\s*([\]\}])", r"\1", cand)
        try:
            return json.loads(cand2)
        except:
            return None

In [ ]:
FEWSHOT_EXAMPLE = r"""
FEW-SHOT EXAMPLE (how to cover UI + backend from code context):

If code has:
- Button confirmBt with setOnClickListener
- Reads EditText newMail + confirmNewMail
- Branch: empty/mismatch -> Toast error
- Calls updateUserEmail(email)
- Backend: getCurrentUser() null -> Toast unauthenticated
- Backend: verifyBeforeUpdateEmail success -> save SharedPreferences + navigate
- Backend: verifyBeforeUpdateEmail failure -> Toast failure

Then tests must include covered_methods like:
["onCreateView","updateUserEmail"] and steps for each branch.
""".strip()

def build_generation_prompt(context: str,
                            file_rel: str,
                            must_cover_methods: List[str],
                            ui_actions: List[Dict[str, str]],
                            retry_missing: Optional[List[str]] = None) -> List[Dict[str, str]]:

    system = (
        "You are a senior SDET and software test designer. "
        "Generate concrete, code-faithful test cases ONLY from the given code context. "
        "Do NOT invent features, methods, screens, APIs, or validations not shown. "
        "You MUST cover every method in MUST_COVER at least once (1 test per method minimum). "
        "If there are UI/CTA signals, include interaction tests relevant to those signals. "
        "Output ONLY a valid JSON array. No markdown. No extra text."
    )

    must_cover_block = "\n".join([f"- {m}" for m in must_cover_methods]) if must_cover_methods else "- (none)"

    ui_block = ""
    if ui_actions:
        ui_lines = []
        for a in ui_actions[:40]:
            ui_lines.append(f"- {a.get('kind')} line {a.get('start_line')}: {a.get('evidence')}")
        ui_block = "\n".join(ui_lines)
    else:
        ui_block = "- (none)"

    retry_block = ""
    if retry_missing:
        retry_block = "\nMISSING METHODS FROM PREVIOUS OUTPUT (ADD TESTS FOR THESE NOW):\n" + \
                      "\n".join([f"- {m}" for m in retry_missing])

    user = f"""
TARGET FILE: {file_rel}

MUST_COVER (ensure each appears in covered_methods at least once):
{must_cover_block}

UI/CTA SIGNALS (use if present):
{ui_block}

OUTPUT FORMAT (JSON array ONLY; EXACT keys):
[
  {{
    "test_case_id": "TC-<Module>-###",
    "title": "...",
    "module": "...",
    "source_files": ["{file_rel}"],
    "covered_methods": ["methodName1","methodName2"],
    "preconditions": ["..."],
    "test_data": ["..."],
    "test_steps": ["..."],
    "expected_result": ["..."]
  }}
]

RULES:
- Keep total test cases <= {MAX_TESTS_PER_FILE} (merge coverage when needed).
- Every MUST_COVER method must be in covered_methods somewhere.
- Steps & expected_result must match code branches, UI actions, and backend outcomes visible.
- Output ONLY the JSON array.

{FEWSHOT_EXAMPLE}
{retry_block}

CONTEXT:
{context}
""".strip()

    return [{"role":"system","content":system},{"role":"user","content":user}]

def chat_generate(tok, model, messages: List[Dict[str, str]], max_new_tokens: int = 1800) -> str:
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.pad_token_id
        )
    return tok.decode(out[0], skip_special_tokens=True)

In [ ]:
def make_safe_name(s: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(s))
    s = re.sub(r"_+", "_", s).strip("._")
    return s or "item"

def write_json(fp: str, obj: Any):
    with open(fp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def as_json_list(x):
    return json.dumps(x if isinstance(x, list) else [], ensure_ascii=False)

def get_top_level_code_dirs(base_dir: str):
    top_dirs = []
    for name in sorted(os.listdir(base_dir)):
        full_path = os.path.join(base_dir, name)
        if os.path.isdir(full_path):
            top_dirs.append((name, full_path))
    return top_dirs

In [ ]:
def run_phase3_single(run_id: int):
    run_root = os.path.join(PHASE3_ROOT, f"run_{run_id}")
    os.makedirs(run_root, exist_ok=True)

    RUN_OUT_DIR = os.path.join(run_root, "CODE_TESTCASES_FILEWISE")
    TRACKING_DIR = os.path.join(run_root, "inference_tracking")

    os.makedirs(RUN_OUT_DIR, exist_ok=True)
    os.makedirs(TRACKING_DIR, exist_ok=True)

    tracker = make_tracker(
        output_dir=TRACKING_DIR,
        project_name="phase3_code_validation_inference",
        output_file=f"phase3_run_{run_id}_inference_emissions.csv"
    )

    print(f"\nPHASE 3 RUN {run_id} INFERENCE TRACKER START")
    tracker.start()
    infer_t0 = time.time()

    top_level_projects = get_top_level_code_dirs(BASE_DIR)

    print(f"\nROOT: {BASE_DIR}")
    print(f"Detected top-level code folders: {len(top_level_projects)}")
    for proj in top_level_projects[:20]:
        print(" -", proj[0])

    summary_rows = []
    master_rows = []

    for proj_idx, (project_name, project_path) in enumerate(top_level_projects, start=1):
        safe_project_name = make_safe_name(project_name)
        project_out_dir = os.path.join(RUN_OUT_DIR, safe_project_name)
        os.makedirs(project_out_dir, exist_ok=True)

        print(f"\n========== PROJECT: {project_name} ==========")
        print(f"Project path: {project_path}")

        units = find_code_units(project_path)

        if not units:
            print(f"  ⚠️ No code units found inside {project_name}")
            continue

        print(f"  Detected units: {len(units)}")
        for u in units[:10]:
            print("   -", u)

        for uidx, (unit_name, unit_path) in enumerate(units, start=1):
            unit_tag = f"{unit_name}_{uidx:02d}"

            code_files = collect_code_files(unit_path)
            print(f"\n  [{unit_tag}] unit_path={unit_path}")
            print(f"    code files found: {len(code_files)}")
            if code_files[:5]:
                print("    sample files:")
                for s in code_files[:5]:
                    print("     -", relpath(s, project_path))

            for fidx, fp in enumerate(code_files, start=1):
                t0 = time.time()

                relf = relpath(fp, project_path)
                file_base = os.path.splitext(os.path.basename(fp))[0]

                safe_unit = make_safe_name(unit_tag)
                safe_file = make_safe_name(file_base)
                file_folder_name = f"{safe_unit}__{safe_file}_{fidx:03d}"

                file_out_dir = os.path.join(project_out_dir, file_folder_name)
                os.makedirs(file_out_dir, exist_ok=True)

                api_map = file_api_map(fp, project_path)

                api_fp = os.path.join(file_out_dir, "api_map.json")
                raw_fp = os.path.join(file_out_dir, "testcases_raw.txt")
                csv_fp = os.path.join(file_out_dir, "testcases.csv")

                write_json(api_fp, api_map)

                must_cover = expected_method_names(api_map)
                ui_actions = api_map.get("ui_actions", [])

                if len(must_cover) == 0 and len(ui_actions) == 0:
                    open(raw_fp, "w", encoding="utf-8").write("")
                    pd.DataFrame(columns=[
                        "test_case_id","title","module","source_files","covered_methods",
                        "preconditions","test_data","test_steps","expected_result"
                    ]).to_csv(csv_fp, index=False)

                    secs = round(time.time() - t0, 2)
                    summary_rows.append({
                        "run_id": run_id,
                        "project": project_name,
                        "project_path": project_path,
                        "unit": unit_tag,
                        "unit_path": unit_path,
                        "file": relf,
                        "file_output_dir": file_out_dir,
                        "methods": api_map["stats"]["methods"],
                        "ui": api_map["stats"]["ui"],
                        "tests": 0,
                        "api_map": api_fp,
                        "raw": raw_fp,
                        "csv": csv_fp,
                        "seconds": secs
                    })
                    print(f"    ⚠️ {os.path.basename(fp)} | methods={api_map['stats']['methods']} ui={api_map['stats']['ui']} | tests=0 | {secs}s")
                    continue

                context = pack_context_for_file(api_map, project_path)

                msgs = build_generation_prompt(
                    context=context,
                    file_rel=relf,
                    must_cover_methods=must_cover,
                    ui_actions=ui_actions,
                    retry_missing=None
                )
                out_text = chat_generate(tokenizer, model, msgs, max_new_tokens=2000)
                items = try_parse_json_array_strict(out_text)

                missing = find_missing_methods(api_map, items or [])

                if (items is None) or (len(items) == 0 and len(must_cover) > 0) or (len(missing) > 0):
                    retry_missing = missing if missing else must_cover
                    msgs2 = build_generation_prompt(
                        context=context,
                        file_rel=relf,
                        must_cover_methods=must_cover,
                        ui_actions=ui_actions,
                        retry_missing=retry_missing
                    )
                    out_text2 = chat_generate(tokenizer, model, msgs2, max_new_tokens=2200)
                    items2 = try_parse_json_array_strict(out_text2)
                    if items2 is not None and len(items2) >= (len(items) if items else 0):
                        out_text = out_text2
                        items = items2

                with open(raw_fp, "w", encoding="utf-8") as f:
                    f.write(out_text)

                if items is None:
                    items = []

                rows = []
                for tc in items:
                    rows.append({
                        "test_case_id": tc.get("test_case_id",""),
                        "title": tc.get("title",""),
                        "module": tc.get("module",""),
                        "source_files": as_json_list(tc.get("source_files",[relf])),
                        "covered_methods": as_json_list(tc.get("covered_methods",[])),
                        "preconditions": as_json_list(tc.get("preconditions",[])),
                        "test_data": as_json_list(tc.get("test_data",[])),
                        "test_steps": as_json_list(tc.get("test_steps",[])),
                        "expected_result": as_json_list(tc.get("expected_result",[])),
                    })

                pd.DataFrame(rows, columns=[
                    "test_case_id","title","module","source_files","covered_methods",
                    "preconditions","test_data","test_steps","expected_result"
                ]).to_csv(csv_fp, index=False)

                secs = round(time.time() - t0, 2)
                final_missing = find_missing_methods(api_map, items)
                if len(must_cover) > 0 and len(final_missing) > 0:
                    miss_note = f" (missing {len(final_missing)})"
                else:
                    miss_note = ""

                summary_rows.append({
                    "run_id": run_id,
                    "project": project_name,
                    "project_path": project_path,
                    "unit": unit_tag,
                    "unit_path": unit_path,
                    "file": relf,
                    "file_output_dir": file_out_dir,
                    "methods": api_map["stats"]["methods"],
                    "ui": api_map["stats"]["ui"],
                    "tests": len(rows),
                    "api_map": api_fp,
                    "raw": raw_fp,
                    "csv": csv_fp,
                    "seconds": secs
                })

                for r in rows:
                    r2 = dict(r)
                    r2["run_id"] = run_id
                    r2["project"] = project_name
                    r2["unit"] = unit_tag
                    r2["file"] = relf
                    r2["file_output_dir"] = file_out_dir
                    master_rows.append(r2)

                print(f"    ✅ {os.path.basename(fp)} | methods={api_map['stats']['methods']} ui={api_map['stats']['ui']} | tests={len(rows)} | {secs}s{miss_note}")

        project_summary_fp = os.path.join(project_out_dir, "SUMMARY_ALL_PROJECTS_FILEWISE.csv")
        project_master_fp  = os.path.join(project_out_dir, "MASTER_ALL_PROJECTS_TESTCASES.csv")

        project_summary_rows = [r for r in summary_rows if r["project"] == project_name]
        project_master_rows = [r for r in master_rows if r["project"] == project_name]

        pd.DataFrame(project_summary_rows).to_csv(project_summary_fp, index=False)
        pd.DataFrame(project_master_rows).to_csv(project_master_fp, index=False)

        print(f"\n  ✅ Project summary saved: {project_summary_fp}")
        print(f"  ✅ Project master saved : {project_master_fp}")

    summary_fp = os.path.join(RUN_OUT_DIR, "SUMMARY_ALL_PROJECTS_FILEWISE.csv")
    master_fp  = os.path.join(RUN_OUT_DIR, "MASTER_ALL_PROJECTS_TESTCASES.csv")

    pd.DataFrame(summary_rows).to_csv(summary_fp, index=False)
    pd.DataFrame(master_rows).to_csv(master_fp, index=False)

    inference_emissions = tracker.stop()
    inference_duration = time.time() - infer_t0
    print(f"\nPHASE 3 RUN {run_id} INFERENCE TRACKER STOP")

    inference_summary = {
        "phase": "phase3_code_validation",
        "run_id": run_id,
        "model_id": MODEL_ID,
        "input_base_dir": BASE_DIR,
        "run_output_dir": RUN_OUT_DIR,
        "summary_csv": summary_fp,
        "master_csv": master_fp,
        "inference_duration_sec": inference_duration,
        "inference_emissions_kg": inference_emissions,
    }

    save_json(
        inference_summary,
        os.path.join(TRACKING_DIR, f"phase3_run_{run_id}_inference_summary.json")
    )

    print("\n✅ Summary:", summary_fp)
    print("✅ Master :", master_fp)

    return inference_summary

In [ ]:
all_run_summaries = []

for run_id in range(1, NUM_RUNS + 1):
    print(f"\n==================== PHASE 3 RUN {run_id}/{NUM_RUNS} ====================")
    run_summary = run_phase3_single(run_id)
    all_run_summaries.append(run_summary)

    if run_id < NUM_RUNS:
        print(f"\nCooldown for {COOLDOWN_SECS} seconds...\n")
        time.sleep(COOLDOWN_SECS)

all_run_summaries